Q1. Theory and Concepts
🔹 (a) Explain the concept of batch normalization

Batch Normalization (BN) is a technique used in neural networks to normalize the activations of each layer for every mini-batch during training.

Instead of letting each layer receive inputs with continuously changing distributions, batch normalization ensures that the inputs to a layer have:

zero mean

unit variance (approximately)

This stabilizes the learning process and allows faster and more reliable training.

🔹 (b) Benefits of using batch normalization

Main benefits:

Faster convergence

Allows higher learning rates

Reduces internal covariate shift

Improves training stability

Acts as a mild regularizer

Reduces sensitivity to initialization

🔹 (c) Working principle of batch normalization

For a mini-batch
𝐵
=
{
𝑥
1
,
𝑥
2
,
…
,
𝑥
𝑚
}
B={x
1
	​

,x
2
	​

,…,x
m
	​

}

Step 1 – Compute batch statistics
𝜇
𝐵
=
1
𝑚
∑
𝑥
𝑖
μ
B
	​

=
m
1
	​

∑x
i
	​

𝜎
𝐵
2
=
1
𝑚
∑
(
𝑥
𝑖
−
𝜇
𝐵
)
2
σ
B
2
	​

=
m
1
	​

∑(x
i
	​

−μ
B
	​

)
2
Step 2 – Normalize
𝑥
^
𝑖
=
𝑥
𝑖
−
𝜇
𝐵
𝜎
𝐵
2
+
𝜖
x
^
i
	​

=
σ
B
2
	​

+ϵ
	​

x
i
	​

−μ
B
	​

	​

Step 3 – Learnable scale and shift
𝑦
𝑖
=
𝛾
𝑥
^
𝑖
+
𝛽
y
i
	​

=γ
x
^
i
	​

+β

Where:

γ (gamma) → learnable scale

β (beta) → learnable shift

These parameters allow the network to restore useful representations if needed.

✅ Q2. Implementation (PyTorch Example)

You can copy this entire section into your notebook.

🔹 Dataset and preprocessing

We use a standard image dataset (for example handwritten digits).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    transform=transform,
    download=True
)

Model WITHOUT batch normalization

In [ ]:
class NetWithoutBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

Model WITH batch normalization

In [ ]:
class NetWithBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.bn1 = nn.BatchNorm1d(256)

        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)

        self.fc3 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(x.size(0), -1)

        x = self.relu(self.bn1(self.fc1(x)))
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.fc3(x)

        return x

Training and evaluation functions

In [ ]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0

    for x, y in loader:
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred = out.argmax(dim=1)
        correct += (pred == y).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0

    with torch.no_grad():
        for x, y in loader:
            out = model(x)
            loss = criterion(out, y)

            total_loss += loss.item()
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

Training both models

In [ ]:
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
device = "cpu"

model_no_bn = NetWithoutBN().to(device)
model_bn = NetWithBN().to(device)

criterion = nn.CrossEntropyLoss()
opt_no_bn = optim.Adam(model_no_bn.parameters(), lr=0.001)
opt_bn    = optim.Adam(model_bn.parameters(), lr=0.001)

In [ ]:
epochs = 5

history = {"no_bn":[], "bn":[]}

for epoch in range(epochs):

    tr_loss1, tr_acc1 = train(model_no_bn, train_loader, opt_no_bn, criterion)
    te_loss1, te_acc1 = evaluate(model_no_bn, test_loader, criterion)

    tr_loss2, tr_acc2 = train(model_bn, train_loader, opt_bn, criterion)
    te_loss2, te_acc2 = evaluate(model_bn, test_loader, criterion)

    history["no_bn"].append((tr_loss1, tr_acc1, te_loss1, te_acc1))
    history["bn"].append((tr_loss2, tr_acc2, te_loss2, te_acc2))

    print(f"Epoch {epoch+1}")
    print("No BN :", tr_acc1, te_acc1)
    print("With BN:", tr_acc2, te_acc2)

Comparison (example table you should create in notebook)
Model	Train Accuracy	Test Accuracy	Train Loss	Test Loss
Without BN	lower	lower	higher	higher
With BN	higher	higher	lower	lower

(Your exact values will depend on training.)

🔹 Discussion: Impact of batch normalization

From the experiment we generally observe:

faster reduction in training loss

smoother training curve

better validation accuracy

less fluctuation during training

Batch normalization stabilizes hidden layer distributions, so gradient flow becomes more reliable.

✅ Q3. Experimentation and Analysis
🔹 (a) Effect of different batch sizes

Try:

In [ ]:
batch_size = 16
batch_size = 64
batch_size = 256

batch_size = 16
batch_size = 64
batch_size = 256